# 👑 FINAL　最終 Boss：咖啡店資料分析專題
**Python 冒險之旅 2026**　｜　Day 6（09/05 六）🏰 資料之島　｜　最終 Boss　｜　🏅 500 XP

📖 對應教科書：全書綜合 + 補充教材


### 🎯 這一關你會學到
- 整合 6 天所學，完成從讀取資料到產出圖表與結論的完整專題

### 🧭 闖關方式
1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/python-quest-2026/)。

> 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  Python 冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins

_LEVEL = "FINAL"
_SALT = "python-quest-2026-datama"
_TASKS = ["F-1", "F-2", "F-3", "F-4", "F-5", "F-6"]
_XP_EACH = 83
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_pyquest_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

def 行列表(out):
    return [ln.rstrip() for ln in str(out).splitlines() if ln.strip()]

class _NeedMoreInput(Exception):
    pass

_HIST = builtins.__dict__.setdefault("_pyquest_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_pyquest_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_pyquest_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

def _find_cell(tid):
    marker = "# 🎯 任務 " + tid
    for cell in reversed(_history()):
        if marker in cell:
            lines = [ln for ln in cell.splitlines()
                     if not re.match(r"\s*(檢查|通關密語)\s*\(", ln)]
            return "\n".join(lines)
    return None

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                _plt.show = _orig_show
        return buf.getvalue(), ns
    run.src = src
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _progress():
    done = sum(1 for t in _TASKS if _PASSED.get(t))
    bar = "■" * done + "□" * (len(_TASKS) - done)
    return f"[{bar}] {done}/{len(_TASKS)}"

def 檢查(tid):
    tid = str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    src = _find_cell(tid)
    if src is None:
        print(f"❌ 找不到「# 🎯 任務 {tid}」的程式格。請先執行那一格（並保留第一行的標記），再執行這裡。")
        return
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        result = (False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。")
    except Exception as e:
        tb = traceback.format_exc().strip().splitlines()[-1]
        result = (False, f"程式執行時發生錯誤 → {tb}")
    ok, extra = (result, "") if isinstance(result, bool) else result
    if ok:
        first = not _PASSED.get(tid)
        _PASSED[tid] = True
        print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")
        if all(_PASSED.get(t) for t in _TASKS):
            print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
    else:
        print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
        if extra: print("   💬 " + str(extra))
        if _HINTS.get(tid): print("   💡 提示：" + _HINTS[tid])
        print("   👉 修改程式後，先重新執行任務那一格，再執行這一格。")

def 通關密語():
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_SALT}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：PYQ-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_F_1(run):
    out, ns = run()
    if sorted(ns.get("na_cols", [])) != ['單價', '數量', '金額']: return (False, "有缺值的欄位是 數量、單價、金額。")
    return (int(ns.get("n_dup", -1)) == 2, "重複列數應該是 2。")
任務定義("F-1", _check_F_1, 提示="na[na > 0].index.tolist()；int(df.duplicated().sum())。")

def _check_F_2(run):
    out, ns = run()
    c = ns.get("clean")
    if c is None or c.shape != (887, 9): return (False, f"clean.shape 應該是 (887, 9)，現在是 {None if c is None else c.shape}。")
    return (abs(float(c['金額'].sum()) - 121050) < 1, "清理後金額總和應該是 121050.0。")
任務定義("F-2", _check_F_2, 提示="df.dropna()、.drop_duplicates()、clean[clean['數量'] > 0]。")

def _check_F_3(run):
    out, ns = run()
    br, iq, cr = ns.get("branch_rev"), ns.get("item_qty"), ns.get("cat_rev")
    if br is None or br.index[0] != '信義店' or abs(float(br.iloc[0]) - 45970) > 1: return (False, "branch_rev 第一名應該是信義店 45970。")
    if iq is None or iq.index[0] != '拿鐵': return (False, "item_qty 第一名應該是拿鐵。")
    return (cr is not None and abs(float(cr.get('烘焙', 0)) - 31525) > -1 and abs(float(cr.get('咖啡', 0)) - 67925) < 1, "cat_rev 的咖啡應該是 67925。")
任務定義("F-3", _check_F_3, 提示="clean.groupby('分店')['金額'].sum().sort_values(ascending=False)。")

def _check_F_4(run):
    out, ns = run()
    dr, dl = ns.get("day_rev"), ns.get("daily")
    if dr is None or dr.index[0] != '星期日': return (False, "最佳星期應該是星期日。")
    if dl is None or len(dl) != 31: return (False, "daily 應該有 31 天。")
    return (str(dl.idxmax()) == '2026-07-25', "營收最高的日期是 2026-07-25。")
任務定義("F-4", _check_F_4, 提示="clean.groupby('日期')['金額'].sum().sort_index()。")

def _check_F_5(run):
    import os
    for f in ('daily.png', 'cat.png'):
        if os.path.exists(f): os.remove(f)
    out, ns = run()
    import matplotlib.pyplot as plt
    plt.close('all')
    return (os.path.exists('daily.png') and os.path.exists('cat.png'), "要產生 daily.png 與 cat.png 兩個圖檔。")
任務定義("F-5", _check_F_5, 提示="kind='line' 與 kind='pie'。")

def _check_F_6(run):
    out, ns = run()
    c = ns.get("結論", {})
    want = {"最賺錢分店": "信義店", "最熱銷品項": "拿鐵", "營收最高類別": "咖啡", "最佳星期": "星期日", "最佳日期": "2026-07-25", "七月總營收": 121050}
    wrong = [k for k in want if c.get(k) != want[k]]
    return (not wrong, f"這些結論不對：{wrong}")
任務定義("F-6", _check_F_6, 提示="item_qty.index[0]、cat_rev.idxmax()、day_rev.index[0]。")


## 👑 最終 Boss：咖啡店資料分析專題
你是「勇者咖啡」的資料分析師。老闆給你 7 月份三家分店的銷售紀錄（有一點髒資料），想知道：

1. 哪一家分店最賺錢？
2. 哪個品項最熱銷？哪個類別貢獻最多營收？
3. 一週中哪一天生意最好？
4. 每天的營收走勢長什麼樣？

資料：`https://raw.githubusercontent.com/johnnychao/python-quest-2026/main/data/coffee_sales.csv`（欄位：訂單編號、日期、星期、分店、品項、類別、數量、單價、金額）

> 這是 6 天課程的總驗收：讀檔、清理、統計、畫圖、下結論。每一步都在前面的關卡出現過。加油，勇者！

In [ ]:
#@title 🈶 中文字型設定（畫圖要顯示中文時先執行；約 20～40 秒）
import subprocess, glob, matplotlib
from matplotlib import font_manager
subprocess.run("apt-get -qq install -y fonts-noto-cjk > /dev/null 2>&1", shell=True)
for f in glob.glob('/usr/share/fonts/opentype/noto/NotoSansCJK*-Regular.ttc'):
    font_manager.fontManager.addfont(f)
matplotlib.rcParams['font.family'] = 'Noto Sans CJK JP'
matplotlib.rcParams['axes.unicode_minus'] = False
print("✅ 中文字型設定完成")

### 🎯 任務 F-1　載入與初探

讀取 CSV 成 `df`，印出形狀、欄位名稱，用 `df.isna().sum()` 找出**有缺值的欄位**，把有缺值的欄位名稱存成串列 `na_cols`，再用 `df.duplicated().sum()` 算出重複列數 `n_dup`。

In [ ]:
# 🎯 任務 F-1　載入與初探（請保留這一行）
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/johnnychao/python-quest-2026/main/data/coffee_sales.csv")
print(df.shape)
print(df.columns.tolist())
na = df.isna().sum()
na_cols = ???
n_dup = ???
print("有缺值的欄位：", na_cols)
print("重複列數：", n_dup)

In [ ]:
檢查("F-1")   # ◀ 執行這一格，看看任務 F-1 有沒有過關

### 🎯 任務 F-2　資料清理

建立乾淨的 `clean`：(1) 刪除有缺值的列 `dropna()`；(2) 刪除重複列 `drop_duplicates()`；(3) 只保留 `數量 > 0` 的列。印出 `clean.shape`（應為 (887, 9)）與金額總和。

In [ ]:
# 🎯 任務 F-2　資料清理（請保留這一行）
clean = df.???
clean = clean.???
clean = clean[???]
print(clean.shape, clean['金額'].sum())

In [ ]:
檢查("F-2")   # ◀ 執行這一格，看看任務 F-2 有沒有過關

### 🎯 任務 F-3　分店與品項分析

用 `clean` 算出：`branch_rev`（各分店金額總和，由高到低）、`item_qty`（各品項數量總和，由高到低）、`cat_rev`（各類別金額總和）。印出三者。

In [ ]:
# 🎯 任務 F-3　分店與品項分析（請保留這一行）
branch_rev = ???
item_qty = ???
cat_rev = ???
print(branch_rev, item_qty, cat_rev, sep="\n\n")

In [ ]:
檢查("F-3")   # ◀ 執行這一格，看看任務 F-3 有沒有過關

### 🎯 任務 F-4　星期與每日走勢

算出 `day_rev`（各星期金額總和，由高到低）與 `daily`（每個日期的金額總和，依日期排序）。印出生意最好的星期與營收最高的那一天：`最佳星期：星期日`、`最佳日期：2026-07-25`。

In [ ]:
# 🎯 任務 F-4　星期與每日走勢（請保留這一行）
day_rev = ???
daily = ???
print("最佳星期：", day_rev.index[0])
print("最佳日期：", daily.idxmax())

In [ ]:
檢查("F-4")   # ◀ 執行這一格，看看任務 F-4 有沒有過關

### 🎯 任務 F-5　視覺化儀表板

畫兩張圖：(1) `daily` 的**線條圖**（標題 `7 月每日營收走勢`，marker `'o'`）並存成 `daily.png`；(2) `cat_rev` 的**圓餅圖**（`plot(kind='pie', autopct='%.1f%%', title='類別營收占比')`）並存成 `cat.png`。

In [ ]:
# 🎯 任務 F-5　視覺化儀表板（請保留這一行）
import matplotlib.pyplot as plt
daily.plot(kind=???, marker='o', title='7 月每日營收走勢', figsize=(10, 4))
plt.savefig('daily.png'); plt.show()
cat_rev.plot(kind=???, autopct='%.1f%%', title='類別營收占比')
plt.savefig('cat.png'); plt.show()

In [ ]:
檢查("F-5")   # ◀ 執行這一格，看看任務 F-5 有沒有過關

### 🎯 任務 F-6　給老闆的結論

把分析結果填進字典 `結論`（從前面算出的變數取值，不要用手打），並用迴圈印出每一項。

In [ ]:
# 🎯 任務 F-6　給老闆的結論（請保留這一行）
結論 = {
    "最賺錢分店": branch_rev.index[0],
    "最熱銷品項": ???,
    "營收最高類別": ???,
    "最佳星期": ???,
    "最佳日期": str(daily.idxmax()),
    "七月總營收": int(clean['金額'].sum()),
}
for k, v in 結論.items():
    print(f"{k}：{v}")

In [ ]:
檢查("F-6")   # ◀ 執行這一格，看看任務 F-6 有沒有過關

## 🎤 成果分享（3 分鐘）
用你的結論與圖表，向「老闆」報告：最重要的一個發現是什麼？你會建議老闆做什麼？
（例如：週一、二生意最差，可以推週一二優惠；烘焙類客單價高，可以推套餐。）

## 🌟 進階挑戰（不計分）
1. 加一欄 `時段`（假設訂單編號尾數決定上午／下午），比較不同時段營收。
2. 用 `clean.pivot_table(index='分店', columns='類別', values='金額', aggfunc='sum')` 做交叉表。
3. 把結論寫成 `report.txt`，附上圖檔，下載給老闆。

---
## 🔑 通關密語
　完成這一關，你就正式從新手村畢業了！
全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**你已經抵達最後一關！回入口網頁領取結業證書吧 🎓**

回到入口網頁：https://johnnychao.github.io/python-quest-2026/